# 🏎️ F1 Pit Stop Prediction | RealMLP + Full Feature Engineering
### Playground Series S6E5

**RealMLP_TD with exact hyperparameters from realmlp-encoding-fe notebook | 6-Fold StratifiedKFold OOF**

- Full tyre physics feature engineering
- NN encoding layer (floor-to-cat, count encoding, quantile binning, combo cats)
- Target encoding on combo features inside each fold (no leakage)
- 6-fold StratifiedKFold — OOF directly stackable with XGBoost OOF

## 1. Setup

In [1]:
%%time
!pip install -q pytabkit

import random
import warnings
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder
import torch
from pytabkit import RealMLP_TD_Classifier

warnings.filterwarnings('ignore')

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)
print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 2.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 55.3 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incom

## 2. Config

In [2]:
class CFG:
    FOLDS  = 6
    SEED   = 42
    TARGET = 'PitNextLap'
    ID     = 'id'

# ── Exact hyperparameters from realmlp-encoding-fe notebook ──
PARAMS = {
    'random_state'   : 42,
    'verbosity'      : 2,
    'val_metric_name': '1-auc_ovr',   # trains directly on AUC, not a proxy loss

    # ── Architecture ──
    'n_ens'          : 24,             # 24 internal ensemble members
    'hidden_sizes'   : [512, 256, 128],
    'act'            : 'silu',
    'embedding_size' : 6,
    'max_one_hot_cat_size': 18,

    # ── Training ──
    'n_epochs'       : 8,
    'batch_size'     : 256,
    'lr'             : 0.03,
    'wd'             : 0.018,
    'sq_mom'         : 0.98,
    'lr_sched'       : 'lin_cos_log_15',
    'first_layer_lr_factor': 0.25,

    # ── Regularization ──
    'p_drop'         : 0.05,
    'p_drop_sched'   : 'expm4t',
    'ls_eps'         : 0.01,
    'ls_eps_sched'   : 'sqrt_cos',

    # ── PBLD (Periodic Basis with Learned Decay) embeddings ──
    'plr_hidden_1'   : 16,
    'plr_hidden_2'   : 8,
    'plr_act_name'   : 'gelu',
    'plr_lr_factor'  : 0.1151,
    'plr_sigma'      : 2.33,

    # ── Preprocessing pipeline ──
    'add_front_scale': False,
    'bias_init_mode' : 'neg-uniform-dynamic-2',
    'tfms'           : ['one_hot', 'median_center', 'robust_scale',
                        'smooth_clip', 'embedding', 'l2_normalize'],

    # ── Early stopping (disabled — fixed epochs) ──
    'use_early_stopping'                    : False,
    'early_stopping_additive_patience'      : 10,
    'early_stopping_multiplicative_patience': 1,
}

## 3. Load Data

In [3]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e5/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/playground-series-s6e5/test.csv')
orig  = pd.read_csv('/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv')

orig.drop(columns=['Normalized_TyreLife'], inplace=True, errors='ignore')

print(f"Train : {train.shape}")
print(f"Test  : {test.shape}")
print(f"Orig  : {orig.shape}")
print(f"Target rate train : {train[CFG.TARGET].mean():.4f}")
print(f"Target rate orig  : {orig[CFG.TARGET].mean():.4f}")

Train : (439140, 16)
Test  : (188165, 15)
Orig  : (101371, 15)
Target rate train : 0.1990
Target rate orig  : 0.2548


## 4. Feature Engineering

Tyre physics + race-progress signals + interaction terms.

In [4]:
COMPOUND_STINT_MEDIANS = {
    'SOFT': 14.0, 'MEDIUM': 17.0, 'HARD': 23.0,
    'INTERMEDIATE': 16.0, 'WET': 14.0
}
COMPOUND_HARDNESS = {
    'SOFT': 1, 'MEDIUM': 2, 'HARD': 3,
    'INTERMEDIATE': 1, 'WET': 0
}

def engineer_features(df):
    df = df.copy()

    # ── Core tyre features ───────────────────────────────────────────────────
    df['ExpectedStint']       = df['Compound'].map(COMPOUND_STINT_MEDIANS).fillna(17.0)
    df['TyreLife_Normalized'] = df['TyreLife'] / df['ExpectedStint']
    df['TyreLife_sq']         = df['TyreLife'] ** 2
    df['TyreLife_sqrt']       = np.sqrt(df['TyreLife'])
    df['TyreLife_log1p']      = np.log1p(df['TyreLife'])
    df['Compound_Hardness']   = df['Compound'].map(COMPOUND_HARDNESS).fillna(2).astype(int)
    df['TyreLife_x_Hardness'] = df['TyreLife'] * df['Compound_Hardness']
    df['Norm_x_Hardness']     = df['TyreLife_Normalized'] * df['Compound_Hardness']

    # ── Tyre age flags ───────────────────────────────────────────────────────
    df['Is_Fresh']      = (df['TyreLife'] <= 3).astype(np.int8)
    df['Is_Old']        = (df['TyreLife'] > 20).astype(np.int8)
    df['Is_VeryOld']    = (df['TyreLife'] > 40).astype(np.int8)
    df['Is_FirstStint'] = (df['Stint'] == 1).astype(np.int8)

    # ── Degradation rate ─────────────────────────────────────────────────────
    df['DegRate'] = df['Cumulative_Degradation'] / (df['TyreLife'] + 1)

    # ── Race progress flags ──────────────────────────────────────────────────
    df['Early_Race']    = (df['RaceProgress'] < 0.25).astype(np.int8)
    df['Late_Race']     = (df['RaceProgress'] >= 0.75).astype(np.int8)
    df['VeryLate_Race'] = (df['RaceProgress'] >= 0.90).astype(np.int8)

    # ── Year flags ───────────────────────────────────────────────────────────
    df['Is_2023'] = (df['Year'] == 2023).astype(np.int8)
    df['Is_2025'] = (df['Year'] == 2025).astype(np.int8)

    # ── Ratio / interaction features ─────────────────────────────────────────
    df['TyreLife_per_Lap']   = df['TyreLife'] / df['LapNumber'].clip(lower=1)
    df['Lap_x_RaceProgress'] = df['LapNumber'] * df['RaceProgress']
    df['Stint_x_TyreLife']   = df['Stint'] * df['TyreLife']
    df['Stint_x_Normalized'] = df['Stint'] * df['TyreLife_Normalized']

    # ── LapTime signal ───────────────────────────────────────────────────────
    df['LapDelta_sq']  = df['LapTime_Delta'] ** 2
    df['LapDelta_abs'] = df['LapTime_Delta'].abs()

    return df

train = engineer_features(train)
test  = engineer_features(test)
orig  = engineer_features(orig)

y_orig = orig[CFG.TARGET].copy()
orig   = orig.drop(columns=[CFG.TARGET])

y        = train[CFG.TARGET].copy()
train_id = train[CFG.ID].copy()
test_id  = test[CFG.ID].copy()

X      = train.drop(columns=[CFG.ID, CFG.TARGET, 'ExpectedStint'])
X_test = test.drop(columns=[CFG.ID, 'ExpectedStint'])
orig   = orig.drop(columns=['ExpectedStint'], errors='ignore')
orig   = orig.reindex(columns=X.columns, fill_value=0)

print(f"X      : {X.shape}")
print(f"X_test : {X_test.shape}")
print(f"orig   : {orig.shape}")

X      : (439140, 37)
X_test : (188165, 37)
orig   : (101371, 37)


## 5. Encoding Layer

Note: RealMLP handles string columns as categoricals internally — **no need to cast to `category` dtype** unlike XGBoost. Columns left as `str` / `object` are fine.

In [5]:
category_map     = {}
IMPORTANT_COMBOS = [('Race', 'Compound'), ('Race', 'Year'), ('Driver', 'Compound')]
NUM_COLS_BASE    = ['LapNumber', 'Stint', 'TyreLife', 'Position',
                    'LapTime (s)', 'RaceProgress', 'Year', 'PitStop']

def feature_engineering_nn(df, fit=False):
    df = df.copy()

    # ── Arithmetic interactions ──────────────────────────────────────────────
    df['_TyreLife_div_LapNumber'] = (
        df['TyreLife'] / df['LapNumber'].clip(lower=1)
    ).astype('float32')
    df['_LapNumber_div_RaceProgress'] = (
        df['LapNumber'] / (df['RaceProgress'] + 1e-6)
    ).astype('float32')

    extra_num = ['_TyreLife_div_LapNumber', '_LapNumber_div_RaceProgress']

    # ── Floor numericals → string categories ────────────────────────────────
    # RealMLP sees these as a second view of numerical features via its embedding path
    for col in NUM_COLS_BASE + extra_num:
        cat_name = f"{col}_cat_"
        if col in df.columns:
            if fit:
                codes, uniques = np.floor(df[col]).factorize()
                category_map[col] = uniques
            else:
                uniques  = category_map.get(col, np.array([]))
                code_map = {cat: i for i, cat in enumerate(uniques)}
                codes    = np.floor(df[col]).map(code_map).fillna(-1).astype('int32')
            df[cat_name] = codes.astype(str)   # string → RealMLP treats as cat

    # ── Count encoding ───────────────────────────────────────────────────────
    for col in ['Driver', 'Compound', 'Race', 'Year']:
        count_name = f"_{col}_count"
        if col in df.columns:
            if fit:
                count_map = df[col].astype(str).value_counts()
                category_map[count_name] = count_map
            else:
                count_map = category_map.get(count_name, pd.Series(dtype=int))
            df[count_name] = df[col].astype(str).map(count_map).fillna(0).astype('int32')

    # ── RaceProgress quantile binning (200 bins) ─────────────────────────────
    bin_name = 'RaceProgress_200_quantile_bin_'
    if fit:
        kb = KBinsDiscretizer(n_bins=200, encode='ordinal',
                              strategy='quantile', subsample=None)
        binned = kb.fit_transform(df[['RaceProgress']]).ravel().astype('int32')
        category_map[bin_name] = kb
    else:
        kb     = category_map.get(bin_name)
        binned = (kb.transform(df[['RaceProgress']]).ravel().astype('int32')
                  if kb else np.zeros(len(df), dtype='int32'))
    df[bin_name] = binned.astype(str)

    # ── Combo categorical features ───────────────────────────────────────────
    combo_names = []
    for cols in IMPORTANT_COMBOS:
        combo_name = '_'.join(cols) + '_combo_'
        combo_names.append(combo_name)
        combo_series = df[cols[0]].astype(str)
        for col in cols[1:]:
            combo_series = combo_series + '_' + df[col].astype(str)
        if fit:
            codes, uniques = pd.factorize(combo_series, sort=False)
            category_map[combo_name] = uniques
        else:
            uniques  = category_map.get(combo_name, np.array([]))
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes    = combo_series.map(code_map).fillna(-1).astype('int32')
        df[combo_name] = codes.astype(str)

    return df, combo_names

X,      combo_names = feature_engineering_nn(X,      fit=True)
X_test, _           = feature_engineering_nn(X_test, fit=False)
orig,   _           = feature_engineering_nn(orig,   fit=False)

orig = orig.reindex(columns=X.columns, fill_value=0)

print(f"X      after encoding : {X.shape}")
print(f"X_test after encoding : {X_test.shape}")
print(f"Combo features        : {combo_names}")

X      after encoding : (439140, 57)
X_test after encoding : (188165, 57)
Combo features        : ['Race_Compound_combo_', 'Race_Year_combo_', 'Driver_Compound_combo_']


## 6. 6-Fold StratifiedKFold Training

In [6]:
%%time
skf        = StratifiedKFold(n_splits=CFG.FOLDS, shuffle=True, random_state=CFG.SEED)
oof_preds  = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
auc_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n{'='*55}")
    print(f"  FOLD {fold}/{CFG.FOLDS}")
    print(f"{'='*55}")

    X_tr  = X.iloc[tr_idx].copy()
    y_tr  = y.iloc[tr_idx].copy()
    X_val = X.iloc[val_idx].copy()
    y_val = y.iloc[val_idx].copy()
    X_tst = X_test.copy()

    # Add original dataset to this fold's training data
    X_tr = pd.concat([X_tr, orig.reset_index(drop=True)], axis=0).reset_index(drop=True)
    y_tr = pd.concat([y_tr, y_orig.reset_index(drop=True)], axis=0).reset_index(drop=True)

    # ── Target encoding on combo features (inside fold → no leakage) ─────────
    te = TargetEncoder(cv=CFG.FOLDS, smooth='auto', shuffle=True, random_state=CFG.SEED)
    tr_enc  = te.fit_transform(X_tr[combo_names], y_tr)
    val_enc = te.transform(X_val[combo_names])
    tst_enc = te.transform(X_tst[combo_names])

    te_names = [f"_{c}_TE" for c in combo_names]
    X_tr[te_names]  = tr_enc
    X_val[te_names] = val_enc
    X_tst[te_names] = tst_enc

    if fold == 1:
        print(f"Total features : {X_tr.shape[1]}")
        print(f"Train rows     : {len(X_tr):,}  (synthetic + original)")
        print(f"Val rows       : {len(X_val):,}")
        print(f"Pos rate train : {y_tr.mean():.4f}  |  val: {y_val.mean():.4f}")

    # ── Train RealMLP ─────────────────────────────────────────────────────────
    model = RealMLP_TD_Classifier(**PARAMS)
    model.fit(X_tr, y_tr, X_val, y_val)

    val_preds       = model.predict_proba(X_val)[:, 1]
    fold_test_preds = model.predict_proba(X_tst)[:, 1]

    oof_preds[val_idx] = val_preds
    test_preds        += fold_test_preds / CFG.FOLDS

    fold_auc = roc_auc_score(y_val, val_preds)
    auc_scores.append(fold_auc)
    print(f"\nFold {fold} AUC: {fold_auc:.6f}")
    torch.cuda.empty_cache()


  FOLD 1/6
Total features : 60
Train rows     : 467,321  (synthetic + original)
Val rows       : 73,190
Pos rate train : 0.2111  |  val: 0.1990
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'TyreLife_Normalized', 'TyreLife_sq', 'TyreLife_sqrt', 'TyreLife_log1p', 'Compound_Hardness', 'TyreLife_x_Hardness', 'Norm_x_Hardness', 'Is_Fresh', 'Is_Old', 'Is_VeryOld', 'Is_FirstStint', 'DegRate', 'Early_Race', 'Late_Race', 'VeryLate_Race', 'Is_2023', 'Is_2025', 'TyreLife_per_Lap', 'Lap_x_RaceProgress', 'Stint_x_TyreLife', 'Stint_x_Normalized', 'LapDelta_sq', 'LapDelta_abs', '_TyreLife_div_LapNumber', '_LapNumber_div_RaceProgress', '_Driver_count', '_Compound_count', '_Race_count', '_Year_count', '_Race_Compound_combo__TE', '_Race_Year_combo__TE', '_Driver_Compound_combo__TE']
Columns classified as categorical: ['Driver', 'Compound', 'Race', 'LapNumbe

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/8: val 1-auc_ovr = 0.054813
Epoch 2/8: val 1-auc_ovr = 0.049705
Epoch 3/8: val 1-auc_ovr = 0.046352
Epoch 4/8: val 1-auc_ovr = 0.045475
Epoch 5/8: val 1-auc_ovr = 0.045975
Epoch 6/8: val 1-auc_ovr = 0.045804
Epoch 7/8: val 1-auc_ovr = 0.045310
Epoch 8/8: val 1-auc_ovr = 0.045818


`Trainer.fit` stopped: `max_epochs=8` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]



Fold 1 AUC: 0.954690

  FOLD 2/6
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'TyreLife_Normalized', 'TyreLife_sq', 'TyreLife_sqrt', 'TyreLife_log1p', 'Compound_Hardness', 'TyreLife_x_Hardness', 'Norm_x_Hardness', 'Is_Fresh', 'Is_Old', 'Is_VeryOld', 'Is_FirstStint', 'DegRate', 'Early_Race', 'Late_Race', 'VeryLate_Race', 'Is_2023', 'Is_2025', 'TyreLife_per_Lap', 'Lap_x_RaceProgress', 'Stint_x_TyreLife', 'Stint_x_Normalized', 'LapDelta_sq', 'LapDelta_abs', '_TyreLife_div_LapNumber', '_LapNumber_div_RaceProgress', '_Driver_count', '_Compound_count', '_Race_count', '_Year_count', '_Race_Compound_combo__TE', '_Race_Year_combo__TE', '_Driver_Compound_combo__TE']
Columns classified as categorical: ['Driver', 'Compound', 'Race', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime (s)_cat_', 'RaceProgress_cat_', 'Year_cat_', 

...


Fold 5 AUC: 0.952312

  FOLD 6/6
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'TyreLife_Normalized', 'TyreLife_sq', 'TyreLife_sqrt', 'TyreLife_log1p', 'Compound_Hardness', 'TyreLife_x_Hardness', 'Norm_x_Hardness', 'Is_Fresh', 'Is_Old', 'Is_VeryOld', 'Is_FirstStint', 'DegRate', 'Early_Race', 'Late_Race', 'VeryLate_Race', 'Is_2023', 'Is_2025', 'TyreLife_per_Lap', 'Lap_x_RaceProgress', 'Stint_x_TyreLife', 'Stint_x_Normalized', 'LapDelta_sq', 'LapDelta_abs', '_TyreLife_div_LapNumber', '_LapNumber_div_RaceProgress', '_Driver_count', '_Compound_count', '_Race_count', '_Year_count', '_Race_Compound_combo__TE', '_Race_Year_combo__TE', '_Driver_Compound_combo__TE']
Columns classified as categorical: ['Driver', 'Compound', 'Race', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime (s)_cat_', 'RaceProgress_cat_', 'Year_cat_', 

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/8: val 1-auc_ovr = 0.054939
Epoch 2/8: val 1-auc_ovr = 0.050087
Epoch 3/8: val 1-auc_ovr = 0.046753
Epoch 4/8: val 1-auc_ovr = 0.046013
Epoch 5/8: val 1-auc_ovr = 0.046511
Epoch 6/8: val 1-auc_ovr = 0.046383
Epoch 7/8: val 1-auc_ovr = 0.045879
Epoch 8/8: val 1-auc_ovr = 0.046533


`Trainer.fit` stopped: `max_epochs=8` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]



Fold 6 AUC: 0.954121
CPU times: user 23min 39s, sys: 3min 44s, total: 27min 23s
Wall time: 27min 4s


## 7. Results

In [7]:
oof_auc = roc_auc_score(y, oof_preds)
print(f"\n{'='*45}")
print(f"  Per-fold AUCs : {[f'{s:.4f}' for s in auc_scores]}")
print(f"  Mean fold AUC : {np.mean(auc_scores):.6f}")
print(f"  OOF ROC-AUC   : {oof_auc:.6f}")
print(f"{'='*45}")


  Per-fold AUCs : ['0.9547', '0.9520', '0.9521', '0.9534', '0.9523', '0.9541']
  Mean fold AUC : 0.953109
  OOF ROC-AUC   : 0.953105


## 8. Save OOF + Submission

In [8]:
oof_df = pd.DataFrame({
    CFG.ID    : train_id.values,
    CFG.TARGET: oof_preds,
})
oof_df.to_csv('oof_realmlp_fe.csv', index=False)
print(f"OOF saved: oof_realmlp_fe.csv  ({len(oof_df):,} rows)")

sub = pd.DataFrame({
    CFG.ID    : test_id.values,
    CFG.TARGET: test_preds,
})
sub.to_csv('submission_realmlp_fe.csv', index=False)
print(f"Submission saved: submission_realmlp_fe.csv")
print(sub.head())

OOF saved: oof_realmlp_fe.csv  (439,140 rows)
Submission saved: submission_realmlp_fe.csv
       id  PitNextLap
0  439140    0.005546
1  439141    0.019512
2  439142    0.006712
3  439143    0.312127
4  439144    0.785838
